In [1]:
############################################################
# 1. 환경 설정 및 라이브러리 로드 
# ----------------------------------------------------------
# - 클리닝 단계에서 사용할 라이브러리 로드
# - Google API 키는 .env 파일에서 불러올 예정
############################################################

import os
import re
import time
import requests
import numpy as np
import pandas as pd

from dotenv import load_dotenv  # Google API를 .env 파일에서 불러옴 

In [2]:
############################################################
# 2. 데이터 로드
# ----------------------------------------------------------
# - 크롤링 결과 Raw 파일 로드
# - 서울교통공사 역 주소 데이터 로드
#   → 역 기반 업소 보완용 기준 데이터
############################################################

RAW_PATH = "../data/gmnara_raw_20260119_173732.csv"
ST_PATH  = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"
CODE_PATH = "../data/sigungu_code_info.csv"

df_raw = pd.read_csv(RAW_PATH)

# 서교공 파일은 EUC-KR 인코딩
st_raw = pd.read_csv(ST_PATH, encoding="euc-kr")

print(df_raw.shape, df_raw.columns.tolist())
print(st_raw.shape, st_raw.columns.tolist())

(109, 10) ['source_site', 'crawled_at', 'listing_url', 'detail_url', 'external_id', 'name_raw', 'category_raw', 'address_raw', 'map_url_raw', 'status_raw']
(289, 7) ['연번', '역번호', '호선', '역명', '역전화번호', '도로명주소', '지번주소']


In [3]:
############################################################
# 3. 주소 표준화
# ----------------------------------------------------------
# - 공백 정리
# - '서울', '서울시' → '서울특별시' 통일
# → 이후 gu 파싱 및 지오코딩 안정성 확보 목적
############################################################
def normalize_spaces(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[《》]", " ", x)          # 특수 괄호류 제거
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_seoul_prefix(addr: str) -> str:
    """
    '서울', '서울시' -> '서울특별시'로 통일
    """
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return addr
    a = normalize_spaces(addr)
    a = re.sub(r"^서울시\s+", "서울특별시 ", a)
    a = re.sub(r"^서울\s+", "서울특별시 ", a)
    return a

In [4]:
############################################################
# 4. 역 기준 데이터 정제
# ----------------------------------------------------------
# - 역명 표준화 ('종로3가' → '종로3가역')
# - 환승역 중복 제거
# - 역명 → 도로명주소 매핑 테이블 생성
############################################################
def clean_station_name(name: str) -> str:
    """
    서교공 역명: '종로3가' -> '종로3가역'
    """
    n = normalize_spaces(name)
    n = re.sub(r"\(.*?\)", "", n).strip()  # 혹시 괄호 부가정보가 있으면 제거
    if not n.endswith("역"):
        n += "역"
    return n

st = st_raw.copy()

st["station_std"] = st["역명"].apply(clean_station_name)

# 서교공 도로명주소는 대개 '서울특별시 ...' 형태지만, 안전하게 prefix normalize
st["station_road_addr"] = st["도로명주소"].astype(str).apply(normalize_seoul_prefix)

# 대표 주소 선택(환승역 중복 제거)
st_master = (
    st.groupby("station_std", as_index=False)
      .agg(station_road_addr=("station_road_addr", "first"))
)

station_road_dict = dict(zip(st_master["station_std"], st_master["station_road_addr"]))
station_master_list = st_master["station_std"].tolist()

In [5]:
############################################################
# 5. raw 주소 분리
# ----------------------------------------------------------
# address_raw →
#   1) 실제 도로명 주소(address_std)
#   2) 역 기반 설명(station_detail)
############################################################

df = df_raw.copy()

# name/category는 raw 그대로
df["name"] = df["name_raw"]
df["category"] = df["category_raw"]

def looks_like_address(x: str) -> bool:
    # 이 파일은 대체로 '서울 강남구 ...' 또는 '서울특별시 ...' 형태
    return bool(re.match(r"^(서울|서울시|서울특별시)\s+\S+구\s+", x))

def split_address_station(raw: str):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return (np.nan, np.nan)

    raw = normalize_spaces(raw)

    # 주소가 아닌 케이스: 역 정보만
    if not looks_like_address(raw):
        return (np.nan, raw)

    # '서울 ... (삼성동) 삼성중앙역 ...' 패턴
    if "(" in raw and ")" in raw and raw.index("(") < raw.index(")"):
        before = raw[:raw.index("(")].strip()
        after  = raw[raw.index(")")+1:].strip()
    else:
        before = raw.strip()
        after = ""

    # station_detail 정리: '서울,경기,인천' 같은 서비스 범위 문자열 제거
    if after:
        # 역/출구/도보 등의 키워드가 없고 지역 나열이면 제거
        if (not re.search(r"(역|출구|도보|인근|거리)", after)) and re.search(r"(서울|경기|인천)", after):
            after = ""

    address_std = normalize_seoul_prefix(before) if before else np.nan
    station_detail = after if after else np.nan

    return (address_std, station_detail)

df[["address_std", "station_detail"]] = df["address_raw"].apply(lambda x: pd.Series(split_address_station(x)))

display(df[["address_raw","address_std","station_detail"]].head(15))


,address_raw,address_std,station_detail
0,서울 강남구 봉은사로 지하 501 (삼성동) 삼성중앙역 6번출구 도보 5분,서울특별시 강남구 봉은사로 지하 501,삼성중앙역 6번출구 도보 5분
1,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천",서울특별시 강남구 강남대로 지하 396,NaN
2,서울 강북구 도봉로 지하 50 (미아동) 미아사거리역3번출구,서울특별시 강북구 도봉로 지하 50,미아사거리역3번출구
3,서울 강남구 학동로33길 15 (논현동),서울특별시 강남구 학동로33길 15,NaN
4,서울 강남구 강남대로 328 (역삼동) 강남역3번출구,서울특별시 강남구 강남대로 328,강남역3번출구
5,서울 강남구 일원로 지하 2 (일원동) 대청역 4번출구로 나와서 직진,서울특별시 강남구 일원로 지하 2,대청역 4번출구로 나와서 직진
6,서울 강남구 학동로 지하 102 (논현동) 논혁역7번출구,서울특별시 강남구 학동로 지하 102,논혁역7번출구
7,서울 서초구 강남대로 48-3 (양재동) 양재동,서울특별시 서초구 강남대로 48-3,양재동
8,서울 강남구 강남대로 지하 396 (역삼동) 강남역 부근,서울특별시 강남구 강남대로 지하 396,강남역 부근
9,서울 강남구 강남대로 328 (역삼동) 강남역,서울특별시 강남구 강남대로 328,강남역


In [6]:
############################################################
# 6. 실제 업소 주소 여부 판단
# ----------------------------------------------------------
# - 실제 도로명 주소인지
# - 단순 '역 2번출구 도보 n분' 설명인지 구분
############################################################

def is_station_based_raw(raw: str):
    """
    raw 주소가 실제 업소 주소가 아니라
    '신림역 2번출구 도보2분' 같은 역 기반 설명이면 True 반환
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return False

    t = normalize_spaces(raw)

    # 1) 아예 주소 형식이 아닌 경우
    if not looks_like_address(t):
        return True

    # 2) 주소는 있는데 괄호 뒤가 전부 역 설명인 경우
    if re.search(r"(역|출구|도보)", t) and not re.search(r"\d+로|\d+길|\d+-\d+", t):
        # 도로명 숫자 패턴이 없고 역 키워드가 중심이면
        return True

    return False


df["is_station_based_raw"] = df["address_raw"].apply(is_station_based_raw)

print("역 기반 raw 주소 개수:", df["is_station_based_raw"].sum())


역 기반 raw 주소 개수: 75


In [7]:
############################################################
# 7. 역명 및 출구 파싱
# ----------------------------------------------------------
# station_detail →
#   - station (역명)
#   - station_exit (역 + n번출구)
############################################################

# station_exit 및 station 파싱
def find_station_by_master(text: str):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    t = normalize_spaces(text)

    # 포함되는 역 후보 찾기
    matches = [s for s in station_master_list if s in t]
    if not matches:
        return None

    # 가장 먼저 등장하는 역을 채택
    matches.sort(key=lambda s: t.index(s))
    return matches[0]

def parse_station_fields_with_master(station_detail):
    if station_detail is None or (isinstance(station_detail, float) and np.isnan(station_detail)):
        return (np.nan, np.nan)

    t = normalize_spaces(station_detail)

    # 도보 n분 제거(출구/역명만 남김)
    t2 = re.sub(r"도보\s*\d+\s*분", "", t).strip()

    station = find_station_by_master(t2)
    if not station:
        return (np.nan, np.nan)

    m_exit = re.search(r"(\d+)\s*번\s*출구", t2)
    if m_exit:
        station_exit = f"{station} {m_exit.group(1)}번출구"
    else:
        station_exit = np.nan

    return (station_exit, station)

df[["station_exit", "station"]] = df["station_detail"].apply(lambda x: pd.Series(parse_station_fields_with_master(x)))

display(df[["station_detail","station_exit","station"]].sample(20, random_state=2))

,station_detail,station_exit,station
14,양재역2번출구,양재역 2번출구,양재역
13,강남역,NaN,강남역
21,신림역 5번출구​,신림역 5번출구,신림역
2,미아사거리역3번출구,미아사거리역 3번출구,미아사거리역
96,방이 몽촌토성역 3번출구 도보2분,몽촌토성역 3번출구,몽촌토성역
98,잠실역 10번출구 도보5분,잠실역 10번출구,잠실역
44,공덕역,NaN,공덕역
97,문정역 3번출구 도보3분,문정역 3번출구,문정역
28,신림역 7번출구 도보1분,신림역 7번출구,신림역
35,구로디지털단지역 4번출구,구로디지털단지역 4번출구,구로디지털단지역


In [8]:
############################################################
# 8. 자치구(gu) 추출
# ----------------------------------------------------------
# - address_std에서 '강남구' 등 자치구 파싱
############################################################

# 자치구 파싱
def parse_gu_from_addr(addr: str):
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return np.nan
    a = normalize_seoul_prefix(addr)
    m = re.search(r"서울특별시\s+(\S+구)\b", a)
    if m:
        return m.group(1)
    # 혹시 '서울 강남구' 형태가 남아있다면
    m2 = re.search(r"서울\s+(\S+구)\b", a)
    if m2:
        return m2.group(1)
    return np.nan

# 1차 gu 채우기(상세주소가 있으면 주소에서)
df["gu"] = df["address_std"].apply(parse_gu_from_addr)

In [9]:
############################################################
# 9. Google Geocoding API 설정
############################################################

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY[:6] + "..." if GOOGLE_API_KEY else "❌ 키 로드 실패")

AIzaSy...


In [10]:
############################################################
# 10-1. 지오코딩 설계
# ----------------------------------------------------------
# 1) 실제 업소 주소 기반 지오코딩
# 2) 이후 역 기반 보완
############################################################

def geocode_google(query: str, api_key: str):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": query,
        "key": api_key,
        "language": "ko",
        "region": "kr",
    }
    r = requests.get(url, params=params, timeout=20)
    data = r.json()

    if data.get("status") != "OK" or not data.get("results"):
        return (None, None, None, None)

    top = data["results"][0]
    loc = top["geometry"]["location"]
    lat, lng = loc["lat"], loc["lng"]

    # gu 추출(가능한 범위 내)
    gu = None
    for comp in top.get("address_components", []):
        if "sublocality_level_1" in comp.get("types", []):
            gu = comp.get("long_name")
            break
        if "administrative_area_level_2" in comp.get("types", []):
            gu = comp.get("long_name")

    return (lat, lng, gu, top.get("formatted_address"))

cache = {}

def geocode_cached(query: str, sleep_sec=0.1):
    if query is None or (isinstance(query, float) and np.isnan(query)):
        return (None, None, None, None)
    q = normalize_spaces(query)
    if not q:
        return (None, None, None, None)
    if q in cache:
        return cache[q]

    out = geocode_google(q, GOOGLE_API_KEY)
    cache[q] = out
    time.sleep(sleep_sec)
    return out

def build_station_exit_query(station_exit: str):
    # 서울로 유도
    return f"서울특별시 {station_exit}" if station_exit else None

def build_station_query(station: str):
    # 1) 서교공 도로명주소가 있으면 그걸 최우선 사용
    if station in station_road_dict:
        return station_road_dict[station]
    # 2) 없으면 '서울특별시 + 역명'으로 보완
    return f"서울특별시 {station}" if station else None

In [11]:
############################################################
# 10-2. 지오코딩 실행
############################################################

df["lat"] = np.nan
df["lng"] = np.nan
df["COL_ADM_SE"] = np.nan

# is_valid_location은 지오코딩 성공 여부가 아니라
# "실제 업소 주소인지 여부"로 먼저 결정
df["is_valid_location"] = ~df["is_station_based_raw"]

# 1) 실제 업소 주소인 경우만 address_std로 1차 지오코딩
for i, row in df.iterrows():
    if row["is_valid_location"] and pd.notna(row["address_std"]):
        lat, lng, gu_geo, _ = geocode_cached(row["address_std"])
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 2) 역 기반인 경우는 항상 is_valid_location=False 유지
#    station_exit → station fallback 수행

mask_station_based = df["is_valid_location"] == False

# station_exit 우선
for i, row in df[mask_station_based].iterrows():
    if pd.notna(row["station_exit"]):
        q = build_station_exit_query(row["station_exit"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 여전히 실패 → station (서교공 도로명주소)
mask_need2 = (df["is_valid_location"] == False) & (df["lat"].isna())

for i, row in df[mask_need2].iterrows():
    if pd.notna(row["station"]):
        q = build_station_query(row["station"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]):
                df.at[i, "gu"] = parse_gu_from_addr(q)
            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo


In [12]:
############################################################
# 11. gu 최종 보완 및 결측 점검 (검증용 셀)
# ----------------------------------------------------------
# - 지오코딩/주소 파싱 이후에도 남아있는 gu 결측을 마지막으로 보완
# - 결측 케이스를 표로 출력하여, 파이프라인의 사각지대를 확인
############################################################
mask_gu_missing = df["gu"].isna()
df.loc[mask_gu_missing, "gu"] = df.loc[mask_gu_missing, "address_raw"].apply(parse_gu_from_addr)

print("gu 결측 개수:", df["gu"].isna().sum())
display(df[df["gu"].isna()][["name","address_raw","address_std","station_detail","station_exit","station"]].head(30))

gu 결측 개수: 1


,name,address_raw,address_std,station_detail,station_exit,station
19,핑크스웨디시,신림역 2번출구 도보2분,NaN,신림역 2번출구 도보2분,신림역 2번출구,신림역


In [13]:
############################################################
# 12. COL_ADM_SE 코드 매핑 및 점검 (검증용 셀)
# ----------------------------------------------------------
# - 시군구 코드 테이블을 불러와 서울(11*) 구 코드만 필터링
# - gu(구명) → SIGUNGU_CD를 매핑해 COL_ADM_SE 채움
# - 매핑 결과 결측량을 출력해 품질 확인
############################################################

# COL_ADM_SE 매핑
code_df = pd.read_csv(CODE_PATH, encoding="utf-8", dtype=str)

# 서울(시도코드 11)만 필터
seoul_gu = code_df[code_df["SIGUNGU_CD"].str.startswith("11")].copy()

# 매핑 딕셔너리 생성
gu_code_dict = dict(zip(seoul_gu["SIGUNGU_NM"], seoul_gu["SIGUNGU_CD"]))

# gu 정규화
def normalize_gu(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    # 혹시 "서울특별시 강남구" 형태라면 마지막 토큰만 사용
    if " " in x:
        x = x.split()[-1]
    return x

df["gu"] = df["gu"].apply(normalize_gu)

# 코드 매핑
df["COL_ADM_SE"] = df["gu"].map(gu_code_dict)

# 점검 출력
print("COL_ADM_SE 결측:", df["COL_ADM_SE"].isna().sum())

display(
    df[df["COL_ADM_SE"].isna()][
        ["name","address_std","gu"]
    ].head(20)
)


COL_ADM_SE 결측: 1


,name,address_std,gu
19,핑크스웨디시,NaN,NaN


In [14]:
############################################################
# 13. 역 기반 케이스 진단 리포트 (검증용 셀)
# ----------------------------------------------------------
# - is_valid_location=False(역 기반)만 따로 분리
# - address_std / station_exit / station 보유 여부와
#   최종 위경도(lat) 결측 현황을 요약 출력
# - 역 기반 처리 전략이 충분히 커버하는지 점검
############################################################
df_false = df[df["is_valid_location"] == False].copy()

print("is_valid_location=False 개수:", len(df_false))
print("그 중 address_std 존재:", df_false["address_std"].notna().sum())
print("그 중 station_exit 존재:", df_false["station_exit"].notna().sum())
print("그 중 station 존재:", df_false["station"].notna().sum())
print("그 중 lat 결측:", df_false["lat"].isna().sum())

display(df_false[["name","address_raw","address_std","station_detail","station_exit","station","lat","lng","gu"]].head(30))


is_valid_location=False 개수: 75
그 중 address_std 존재: 71
그 중 station_exit 존재: 37
그 중 station 존재: 61
그 중 lat 결측: 14


,name,address_raw,address_std,station_detail,station_exit,station,lat,lng,gu
0,바나나테라피,서울 강남구 봉은사로 지하 501 (삼성동) 삼성중앙역 6번출구 도보 5분,서울특별시 강남구 봉은사로 지하 501,삼성중앙역 6번출구 도보 5분,삼성중앙역 6번출구,삼성중앙역,37.512889,127.052651,강남구
1,영앤핫 홈케어,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천",서울특별시 강남구 강남대로 지하 396,NaN,NaN,NaN,NaN,NaN,강남구
2,구인마사지,서울 강북구 도봉로 지하 50 (미아동) 미아사거리역3번출구,서울특별시 강북구 도봉로 지하 50,미아사거리역3번출구,미아사거리역 3번출구,미아사거리역,37.613251,127.030084,강북구
4,더원왁싱,서울 강남구 강남대로 328 (역삼동) 강남역3번출구,서울특별시 강남구 강남대로 328,강남역3번출구,강남역 3번출구,강남역,37.497952,127.027619,강남구
5,강남 vip스웨디시&왁싱샵,서울 강남구 일원로 지하 2 (일원동) 대청역 4번출구로 나와서 직진,서울특별시 강남구 일원로 지하 2,대청역 4번출구로 나와서 직진,대청역 4번출구,대청역,37.493601,127.079526,강남구
6,강남일급수스웨디시,서울 강남구 학동로 지하 102 (논현동) 논혁역7번출구,서울특별시 강남구 학동로 지하 102,논혁역7번출구,NaN,NaN,NaN,NaN,강남구
8,썬아로마,서울 강남구 강남대로 지하 396 (역삼동) 강남역 부근,서울특별시 강남구 강남대로 지하 396,강남역 부근,NaN,강남역,37.500078,127.038542,강남구
9,라움테라피,서울 강남구 강남대로 328 (역삼동) 강남역,서울특별시 강남구 강남대로 328,강남역,NaN,강남역,37.500078,127.038542,강남구
10,엠테라피,서울 강남구 강남대로 지하 396 (역삼동) 강남역,서울특별시 강남구 강남대로 지하 396,강남역,NaN,강남역,37.500078,127.038542,강남구
11,네오테라피,서울 강남구 봉은사로 지하 102 (역삼동) 역삼 신논현,서울특별시 강남구 봉은사로 지하 102,역삼 신논현,NaN,NaN,NaN,NaN,강남구


In [15]:
############################################################
# 14. sigu 생성 및 점검 (검증용 셀)
# ----------------------------------------------------------
# - 분석/시각화/통합 단계에서 쓰기 쉽도록
#   sigu = "서울특별시 {gu}" 형태로 생성
# - 중복 제거 리스트 출력로 정상 생성 여부 확인
############################################################

def build_sigu(gu):
    if pd.isna(gu):
        return np.nan
    gu = str(gu).strip()
    if not gu:
        return np.nan
    return f"서울특별시 {gu}"

df["sigu"] = df["gu"].apply(build_sigu)

# 점검
print("sigu 결측:", df["sigu"].isna().sum())
display(df[["gu","sigu"]].drop_duplicates().sort_values("gu").head(20))

sigu 결측: 1


,gu,sigu
0,강남구,서울특별시 강남구
56,강동구,서울특별시 강동구
2,강북구,서울특별시 강북구
63,강서구,서울특별시 강서구
20,관악구,서울특별시 관악구
70,광진구,서울특별시 광진구
26,구로구,서울특별시 구로구
75,금천구,서울특별시 금천구
39,노원구,서울특별시 노원구
40,동대문구,서울특별시 동대문구


In [16]:
############################################################
# 15. Clean 스키마 구성 및 CSV 저장 (산출물 생성 셀)
# ----------------------------------------------------------
# - 모델/분석에 필요한 컬럼만 선별하여 clean 스키마 구성
# - clean_gmnara.csv로 저장 (utf-8-sig)
# - 샘플 head 출력로 저장 전 결과 확인
############################################################

clean_cols = [
    "name",
    "category",
    "address_std",
    "gu",
    "sigu",
    "lat",
    "lng",
    "is_valid_location",
    "COL_ADM_SE",
    "station",
    "station_exit",
    "station_detail",
]

df_clean = df[clean_cols].copy()

# OUT = "../data/clean_gmnara.csv"
OUT = "../test_data/clean_gmnara.csv"    # 시연용
df_clean.to_csv(OUT, index=False, encoding="utf-8-sig")

print("Saved:", OUT)
display(df_clean.head(20))

Saved: ../test_data/clean_gmnara.csv


,name,category,address_std,gu,sigu,lat,lng,is_valid_location,COL_ADM_SE,station,station_exit,station_detail
0,바나나테라피,로드샵,서울특별시 강남구 봉은사로 지하 501,강남구,서울특별시 강남구,37.512889,127.052651,False,11230,삼성중앙역,삼성중앙역 6번출구,삼성중앙역 6번출구 도보 5분
1,영앤핫 홈케어,홈타이,서울특별시 강남구 강남대로 지하 396,강남구,서울특별시 강남구,NaN,NaN,False,11230,NaN,NaN,NaN
2,구인마사지,로드샵,서울특별시 강북구 도봉로 지하 50,강북구,서울특별시 강북구,37.613251,127.030084,False,11090,미아사거리역,미아사거리역 3번출구,미아사거리역3번출구
3,5월스파,로드샵,서울특별시 강남구 학동로33길 15,강남구,서울특별시 강남구,37.515718,127.032294,True,11230,NaN,NaN,NaN
4,더원왁싱,왁싱,서울특별시 강남구 강남대로 328,강남구,서울특별시 강남구,37.497952,127.027619,False,11230,강남역,강남역 3번출구,강남역3번출구
5,강남 vip스웨디시&왁싱샵,로드샵,서울특별시 강남구 일원로 지하 2,강남구,서울특별시 강남구,37.493601,127.079526,False,11230,대청역,대청역 4번출구,대청역 4번출구로 나와서 직진
6,강남일급수스웨디시,로드샵,서울특별시 강남구 학동로 지하 102,강남구,서울특별시 강남구,NaN,NaN,False,11230,NaN,NaN,논혁역7번출구
7,즐겨찾기,로드샵,서울특별시 서초구 강남대로 48-3,서초구,서울특별시 서초구,37.469273,127.039903,True,11220,NaN,NaN,양재동
8,썬아로마,로드샵,서울특별시 강남구 강남대로 지하 396,강남구,서울특별시 강남구,37.500078,127.038542,False,11230,강남역,NaN,강남역 부근
9,라움테라피,로드샵,서울특별시 강남구 강남대로 328,강남구,서울특별시 강남구,37.500078,127.038542,False,11230,강남역,NaN,강남역


In [17]:
############################################################
# 16. 품질 보고 (최종 검증용 셀)
# ----------------------------------------------------------
# - 클리닝 결과의 핵심 지표를 요약 출력
#   (실주소/역기반 분포, 위경도 확보율, gu 결측 등)
############################################################

# 품질 보고
print("총 행:", len(df))
print("실제 업소 주소(TRUE):", df["is_valid_location"].sum())
print("역 기반 주소(FALSE):", (~df["is_valid_location"]).sum())
print("최종 위경도 확보:", df["lat"].notna().sum())
print("위경도 결측:", df["lat"].isna().sum())
print("gu 결측:", df["gu"].isna().sum())


총 행: 109
실제 업소 주소(TRUE): 34
역 기반 주소(FALSE): 75
최종 위경도 확보: 95
위경도 결측: 14
gu 결측: 1
